# BizLens v2.2.12 - Conjoint Analysis (Comprehensive)

Understand customer preferences and trade-offs using **Conjoint Analysis** with BizLens.
We will analyze a simulated smartphone purchase decision dataset.

In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "bizlens"])
print("✅ BizLens v2.2.13 ready for Conjoint Analysis!")

In [ ]:
import bizlens as bl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

sns.set_style("whitegrid")
# bl.set_profiling(True)

# ── Optional: Use a BizLens built-in dataset ─────────────────────────────
# df = bl.load_dataset('tips')  # use tip/total_bill/size as preference attributes
# df = bl.load_dataset('diamonds')  # use cut/color/clarity/price as product attributes
#
# After loading, explore with:
# print(df.shape)         # rows × columns
# print(df.dtypes)        # column types
# print(df.head())        # first 5 rows
#
# ── Export / Save Results ─────────────────────────────────────────────────
# pd.DataFrame(utilities).to_csv('conjoint_utilities.csv', index=False)
# df.to_excel('output.xlsx', index=False)  # Save as Excel
# df.to_json('output.json', orient='records', indent=2)  # Save as JSON
# ── Polars alternative (optional — for users who prefer polars) ──────────────
# import polars as pl
# df_pl = pl.from_pandas(bl.load_dataset('titanic'))    # load → convert to polars
# df_pl = pl.read_csv('your_data.csv')                  # or read directly
# df_pl = pl.DataFrame(data)                             # or create from dict
#
# BizLens accepts both pandas and polars DataFrames transparently:
# bl.tables.frequency_table(df_pl['sex'])               # works with polars too
# bl.quality.data_profile(df_pl)                        # works with polars too
#
# Convert polars back to pandas when needed:
# df = df_pl.to_pandas()

## 1. Simulate Realistic Conjoint Dataset (Smartphones)

In [ ]:
np.random.seed(42)

# Attributes and levels
brands = ['Apple', 'Samsung', 'Google', 'OnePlus']
prices = [699, 899, 1099, 1299]
screens = ['6.1"', '6.4"', '6.7"']
cameras = ['Triple', 'Quad', 'Penta']
batteries = ['4000mAh', '4500mAh', '5000mAh']

# Generate 800 choice observations
n = 800
data = {
    'brand': np.random.choice(brands, n),
    'price': np.random.choice(prices, n),
    'screen_size': np.random.choice(screens, n),
    'camera': np.random.choice(cameras, n),
    'battery': np.random.choice(batteries, n),
    'chosen': np.random.choice([0, 1], n, p=[0.65, 0.35])  # 1 = chosen
}

df = pd.DataFrame(data)
print(f"Simulated conjoint dataset created with {len(df)} observations")
df.head()

## 2. Exploratory Analysis with BizLens

In [ ]:
bl.describe(df)
bl.quality.data_profile(df)

## 3. Conjoint Model (Regression-based Part-Worth Utilities)

In [ ]:
# One-hot encode categorical attributes
df_encoded = pd.get_dummies(df, columns=['brand', 'screen_size', 'camera', 'battery'], drop_first=True)

X = df_encoded.drop('chosen', axis=1)
X = sm.add_constant(X)
y = df_encoded['chosen']

conjoint_model = sm.Logit(y, X).fit(disp=0)
print(conjoint_model.summary())

## 4. Part-Worth Utilities & Attribute Importance

In [ ]:
# Extract part-worth utilities
utilities = conjoint_model.params[1:]
print("\n=== Part-Worth Utilities ===")
print(utilities.sort_values(ascending=False))

# Attribute importance (range of utilities per attribute)
importance = {}
for attr in ['brand', 'screen_size', 'camera', 'battery', 'price']:
    cols = [c for c in utilities.index if c.startswith(attr)]
    if cols:
        importance[attr] = utilities[cols].max() - utilities[cols].min()

importance = pd.Series(importance).sort_values(ascending=False)
print("\n=== Attribute Importance ===")
print(importance)

## 5. Visualizations

In [ ]:
# Part-worth utilities plot
plt.figure(figsize=(12, 6))
utilities.sort_values().plot(kind='barh', color='skyblue')
plt.title('Part-Worth Utilities for Smartphone Attributes')
plt.xlabel('Utility')
plt.show()

In [ ]:
# Attribute importance chart
plt.figure(figsize=(10, 5))
sns.barplot(x=importance.values, y=importance.index, palette='viridis')
plt.title('Relative Importance of Attributes in Purchase Decision')
plt.xlabel('Importance Score')
plt.show()